In [ ]:
# ================================
# 1. IMPORT LIBRARIES
# ================================
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)
from tqdm import tqdm
import joblib


# ================================
# 2. DATASET PATH
# ================================
dataset_path = "/kaggle/input/datasets/pravallikann/combined-anemia/Fingernails"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


# ================================
# 3. IMAGE PREPROCESSING FUNCTION
# ================================
def preprocess_image(image_path):
    """
    Reads and preprocesses a single image.
    Returns: PIL-compatible numpy array (H, W, 3) uint8 in RGB, or None on failure.
    """
    img = cv2.imread(image_path)
    if img is None:
        return None

    # BGR → RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Threshold to remove dark background
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)

    # Find largest contour (fingernail ROI)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None

    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    roi = img_rgb[y:y+h, x:x+w]

    # Resize to EfficientNet-B4 native input size (380x380)
    roi = cv2.resize(roi, (380, 380))

    return roi  # uint8 RGB


# ================================
# 4. PYTORCH DATASET CLASS
# ================================
class AnemiaDataset(Dataset):
    """
    Custom Dataset for anemia fingernail images.
    Applies torchvision transforms (augmentation + normalization).
    """
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = preprocess_image(self.image_paths[idx])

        if img is None:
            # Return a blank image if loading fails
            img = np.zeros((380, 380, 3), dtype=np.uint8)

        # Convert numpy HWC → PIL for torchvision transforms
        from PIL import Image
        img_pil = Image.fromarray(img)

        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = transforms.ToTensor()(img_pil)

        label = self.labels[idx]
        return img_tensor, label


# ================================
# 5. TRANSFORMS (TRAIN & VAL)
# ================================
# ImageNet mean/std — standard for EfficientNet pretrained weights
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


# ================================
# 6. LOAD DATASET & BUILD FILE LISTS
# ================================
image_paths = []
labels      = []
classes     = ["Anemic", "NonAnemic"]

for label, class_name in enumerate(classes):
    class_folder = os.path.join(dataset_path, class_name)
    for img_name in tqdm(os.listdir(class_folder), desc=f"Loading {class_name}"):
        img_path = os.path.join(class_folder, img_name)
        image_paths.append(img_path)
        labels.append(label)

image_paths = np.array(image_paths)
labels      = np.array(labels)

print(f"\nTotal images : {len(image_paths)}")
print(f"Class distribution → Anemic: {(labels==0).sum()}, NonAnemic: {(labels==1).sum()}")


# ================================
# 7. TRAIN-TEST SPLIT (STRATIFIED 80-20)
# ================================
X_train_paths, X_test_paths, y_train, y_test = train_test_split(
    image_paths, labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

print(f"Train samples : {len(X_train_paths)}")
print(f"Test  samples : {len(X_test_paths)}")


# ================================
# 8. DATALOADERS
# ================================
BATCH_SIZE = 16   # Reduce to 8 if GPU OOM

train_dataset = AnemiaDataset(X_train_paths, y_train, transform=train_transform)
test_dataset  = AnemiaDataset(X_test_paths,  y_test,  transform=val_transform)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


# ================================
# 9. BUILD EFFICIENTNET-B4 MODEL
# ================================
def build_efficientnet_b4(num_classes=2, dropout_rate=0.4):
    """
    Loads pretrained EfficientNet-B4 and replaces the classifier head.
    """
    # Load pretrained EfficientNet-B4
    model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)

    # Freeze all backbone layers first
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze last 2 blocks for fine-tuning (blocks 6 & 7)
    for block in list(model.features.children())[-2:]:
        for param in block.parameters():
            param.requires_grad = True

    # Replace classifier head
    in_features = model.classifier[1].in_features   # 1792 for B4
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout_rate, inplace=True),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes)
    )

    return model


model = build_efficientnet_b4(num_classes=2, dropout_rate=0.4)
model = model.to(DEVICE)

# Count trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {trainable:,}")


# ================================
# 10. LOSS, OPTIMIZER & SCHEDULER
# ================================
# Class weights to handle imbalance
class_counts  = np.bincount(y_train)
class_weights = 1.0 / class_counts
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

# Cosine Annealing LR scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)


# ================================
# 11. TRAINING LOOP
# ================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0

    for images, labels_batch in tqdm(loader, desc="Training", leave=False):
        images, labels_batch = images.to(device), labels_batch.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds        = outputs.argmax(dim=1)
        correct      += (preds == labels_batch).sum().item()
        total        += images.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total   = 0
    all_probs  = []
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, labels_batch in tqdm(loader, desc="Evaluating", leave=False):
            images, labels_batch = images.to(device), labels_batch.to(device)

            outputs  = model(images)
            loss     = criterion(outputs, labels_batch)
            probs    = torch.softmax(outputs, dim=1)[:, 1]
            preds    = outputs.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            correct      += (preds == labels_batch).sum().item()
            total        += images.size(0)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_batch.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_probs), np.array(all_labels)


# ================================
# MAIN TRAINING RUN
# ================================
NUM_EPOCHS  = 11
best_val_acc = 0.0
best_model_path = "/kaggle/working/best_efficientnet_b4_anemia.pth"

train_loss_history = []
val_loss_history   = []
train_acc_history  = []
val_acc_history    = []

for epoch in range(1, NUM_EPOCHS + 1):

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, _, _, _ = evaluate(model, test_loader, criterion, DEVICE)

    scheduler.step()

    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)
    train_acc_history.append(train_acc)
    val_acc_history.append(val_acc)

    print(f"Epoch [{epoch:02d}/{NUM_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✅ Best model saved (Val Acc: {best_val_acc:.4f})")


# ================================
# 12. SAVE FINAL MODEL
# ================================
save_path = "/kaggle/working/anemia_efficientnet_b4_final.pth"
torch.save(model.state_dict(), save_path)
print(f"\nFinal model saved at: {save_path}")


# ================================
# 13. LOAD BEST MODEL & PREDICT
# ================================
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()
print("Best model loaded successfully!")

_, _, y_pred, y_prob, y_true = evaluate(model, test_loader, criterion, DEVICE)


# ================================
# 14. EVALUATION METRICS
# ================================
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)
roc_auc   = roc_auc_score(y_true, y_prob)

print("\n===== MODEL PERFORMANCE =====")
print("Accuracy  :", round(accuracy,  4))
print("Precision :", round(precision, 4))
print("Recall    :", round(recall,    4))
print("F1 Score  :", round(f1,        4))
print("ROC-AUC   :", round(roc_auc,   4))

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=classes))


# ================================
# 15. REAL IMAGE PREDICTION
# ================================
def predict_anemia(image_path, model, device=DEVICE):
    """
    Predicts Anemic / Non-Anemic for a single image.
    """
    processed_img = preprocess_image(image_path)

    if processed_img is None:
        print("Image processing failed.")
        return

    from PIL import Image
    img_pil    = Image.fromarray(processed_img)
    img_tensor = val_transform(img_pil).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output      = model(img_tensor)
        probability = torch.softmax(output, dim=1)[0][1].item()
        prediction  = output.argmax(dim=1).item()

    result = "Non-Anemic" if prediction == 1 else "Anemic"

    print("\nPrediction           :", result)
    print("Probability (result):", round(probability, 4))

In [ ]:
def predict_anemia(image_path, model):
    """
    Predicts Anemic / Non-Anemic for a single fingernail image.
    """
    processed_img = preprocess_image(image_path)
    if processed_img is None:
        print("Image processing failed.")
        return

    features = extract_features(processed_img)
    features = np.array(features).reshape(1, -1)

    prediction   = model.predict(features)[0]
    prob_anemic  = model.predict_proba(features)[0][0]   # index 0 → Anemic
    prob_nonanemic = model.predict_proba(features)[0][1] # index 1 → Non-Anemic

    if prediction == 0:
        result      = "Anemic"
        probability = prob_anemic
    else:
        result      = "Non-Anemic"
        probability = prob_nonanemic

    print("\nPrediction              :", result)
    print(f"Probability ({result}):", round(probability, 4))

In [ ]:
# Example usage:
image_path("/kaggle/input/datasets/pravallikann/nnnnnnnn/WhatsApp Image 2026-03-03 at 18.45.26.jpeg",model)
predict_anemia(image_path,model)

In [ ]:
# ================================
# PREDICT ON REAL IMAGE
# ================================
predict_anemia("/kaggle/input/datasets/pravallikann/nnnnnnnn/WhatsApp Image 2026-03-03 at 18.45.26.jpeg", loaded_model)

In [ ]:
# ================================
# PREDICT ON REAL IMAGE (EfficientNet)
# ================================
def predict_anemia(image_path, model, device=DEVICE):

    processed_img = preprocess_image(image_path)
    if processed_img is None:
        print("Image processing failed.")
        return

    from PIL import Image
    img_pil    = Image.fromarray(processed_img)
    img_tensor = val_transform(img_pil).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output         = model(img_tensor)
        probs          = torch.softmax(output, dim=1)[0]
        prediction     = output.argmax(dim=1).item()
        prob_anemic    = probs[0].item()
        prob_nonanemic = probs[1].item()

    if prediction == 0:
        result      = "Anemic"
        probability = prob_anemic
    else:
        result      = "Non-Anemic"
        probability = prob_nonanemic

    print("\nPrediction              :", result)
    print(f"Probability ({result}) :", round(probability, 4))


# Run prediction
predict_anemia("/kaggle/input/datasets/pravallikann/nnnnnnnn/WhatsApp Image 2026-03-03 at 18.45.26.jpeg", model)

In [ ]:
# ================================
# SAVE NOTEBOOK
# ================================
import IPython, json, os

ip   = IPython.get_ipython()
hist = list(ip.history_manager.get_range(output=False))

notebook = {
    "nbformat": 4,
    "nbformat_minor": 5,
    "metadata": {
        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
        "language_info": {"name": "python", "version": "3.12.0"}
    },
    "cells": [
        {
            "cell_type"       : "code",
            "execution_count" : line_num,
            "metadata"        : {},
            "outputs"         : [],
            "source"          : src
        }
        for _, line_num, src in hist if src.strip()
    ]
}

save_path = "/kaggle/working/anemia_efficientnet_b4_notebook.ipynb"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(notebook, f, indent=2, ensure_ascii=False)

print(f"✅ Notebook saved at : {save_path}")
print(f"   Total cells       : {len(notebook['cells'])}")
print(f"   File size         : {round(os.path.getsize(save_path)/1024, 2)} KB")
print("\n👉 Output tab → Download anemia_efficientnet_b4_notebook.ipynb")